In [1]:
# ######### used this part for fixing problems running on ARC #

import os


os.environ['HF_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_HUB_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['XDG_CACHE_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['NB_USER'] = 'ishtiahmed'#'ishtiaqueahmedk'
os.environ['TRANSFORMERS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_DATASETS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'




In [2]:
import json
import os
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, HTML
import re
import copy

from typing import Dict, Any, List



import torch
from PIL import Image

from openai import OpenAI
import argparse

In [3]:
# Modify OpenAI's API key and API base to use the server.
openai_api_key = "sk-7eedef3fad2b4509bc7c0562fc866260"
openai_api_base = "https://llm-api.arc.vt.edu/api/v1"

client = OpenAI(
        api_key=openai_api_key,
        base_url=openai_api_base,
    )

models = client.models.list()
model = models.data[0].id
# print(models.data[0].id)
print(model)

gpt-oss-120b


In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Briefly answer: What is the capital of Virginia?"},
]


chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
    )
response_text = chat_completion.choices[0].message.content
print(f"**Model ({model}) Response:**\n{response_text}")


**Model (gpt-oss-120b) Response:**
Richmond.


In [5]:
def call_arc_llm(prompt: str):
    messages = [
    {"role": "system", "content": "You are a helpful assistant who closely follows rules."},
    {"role": "user", "content": prompt},
]


    chat_completion = client.chat.completions.create(
            messages=messages,
            model=model,
            temperature=0.7
        )
    response_text =  chat_completion.choices[0].message.content    
    return response_text

def call_gpt_api(prompt: str):

    # # For OpenAI:
    import openai
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content


def call_claude_api(prompt: str):

    # # For Anthropic Claude:
    import anthropic
    client = anthropic.Anthropic(api_key="your-api-key")
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text
    

def call_llm(prompt: str) -> str:
    """
    Call your LLM model here.
    Replace this function with your actual LLM call.
        
    # # For OpenAI:
    # outputs =  call_gpt_api(prompt)
    
    # # For Anthropic Claude:
    # outputs =  call_claude_api(prompt)
    """
    # PLACEHOLDER - Replace with your actual LLM call

#     input_text = prompt
# #         print(input_text)
#     input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")
    
#     outputs = model.generate(input_ids)
#     outputs = tokenizer.decode(outputs[0][1:-1])
    
#     print("outputs: ", outputs)

    # raise NotImplementedError("Replace this with your LLM call")


    outputs =  call_arc_llm(prompt)
    # print(f"Response:**\n{outputs}")

    return outputs




In [6]:

def remove_similarity_scores(question: Dict[str, Any]) -> Dict[str, Any]:
    """Remove similarity_scores from a question entry."""
    clean_question = question.copy()
    if 'similarity_scores' in clean_question:
        del clean_question['similarity_scores']
    del clean_question['question']
    
    return clean_question

def create_prompt(question: Dict[str, Any]) -> str:
    """Create the full prompt with the question JSON."""
    # clean_question = remove_similarity_scores(question)
    clean_question = question.copy()


    
    input_json = json.dumps(clean_question, indent=2)
    # print(f"\n\n-------------------------input_json: {input_json}")
    return PROMPT_TEMPLATE.replace("{input_json}", input_json)



In [7]:
def get_map_dict(json_data):
    
    #create dictionary ishti

    captions_mcqid_pair_dict = {}
    class_freq_dict = {}
    
    for mcq in json_data:
        
        options = mcq['options'] # get the answer choices dictionary of the first mcq
        correct_option_description = {options[mcq['correct_answer']]} # get the correct answer caption
        class_name = mcq['mcq_id']
    
        # make sure that there is only one correct caption
        assert len(correct_option_description)==1, "More than one correct answer!!"
    
        #convert to string
        correct_option_description = next(iter(correct_option_description))
    
        if correct_option_description in captions_mcqid_pair_dict: # if caption already in dict
            
            # get the existing mcq_id and make sure it matches
            exist_class_name = captions_mcqid_pair_dict[correct_option_description] 
            
            # make sure mcq_id matches
            assert exist_class_name == class_name, f"mismatch in class name: [{exist_class_name}] and [{class_name}]"
            class_freq_dict [class_name] = class_freq_dict [class_name] + 1
            
    
                
        else:
            captions_mcqid_pair_dict[correct_option_description] = class_name
    
            class_freq_dict [class_name] = 1 

    return captions_mcqid_pair_dict
    
    
    # print(f"Successfully created class_names dictionary with {len(captions_mcqid_pair_dict)} Class entries from JSON file:\n {filename}.\n")
    
    # print(captions_mcqid_pair_dict)
    # print(class_freq_dict)

    

In [8]:
def cleanup_inaturalist(data):
    raise NotImplementedError("Must implement the 'inaturalist cleanup' method.")

---
# Task 4 generation code -- Negation
---

In [9]:


PROMPT_TEMPLATE = """ 
You are given a multiple-choice question about object images. Each option is a natural-language caption describing an object. One option (the correct answer) is the reference object. One of the other options is the target object. The remaining options are distractors.

Your task is to generate a short Final modification caption that describes the target object in terms of its visual similarities and differences relative to the reference object. This caption will be used for a composed image retrieval task, where a model must use BOTH the reference image and the modification caption to find the target.

Follow these instructions carefully:

1. Inputs

   * You will receive:

     * A set of options (e.g., A, B, C, D), each a caption describing an object.
     * A Reference Class and a field `correct_answer` indicating which option is the reference object.
     * A separate indication of which option is the target object.
   * Assume the reference object corresponds to “Image 1”.
   * The other options (besides the target) are distractor objects.

2. Extract only explicit visual attributes

   * For the reference object caption, list all explicitly described visual attributes, such as:

     * Body color and pattern
     * Head and neck color and pattern
     * Bill (beak) color and shape
     * Wing color, pattern, shape, and length
     * Tail shape or special markings
     * Distinctive visual features (e.g., throat pouch, neck band)
     * Visible environment only if explicitly described (e.g., “resting on grassy areas”)
   * For the target object caption, list all explicitly described visual attributes of the same kinds.
   * VERY IMPORTANT: Do NOT invent or add any attribute that is not explicitly mentioned in the captions. Do not infer new colors, shapes, patterns, body parts, background, or behaviors beyond what the text directly states.

3. Similarities between reference and target (relational only)

   * Identify which visual attributes are semantically similar between the reference and target objects, based solely on the explicitly described attributes in both captions
   * If there are no similarities between them, you can skip this Similarities part and go to the next step.
   * When you describe similarities, use purely relational language that points back to the reference object WITHOUT restating the concrete attribute values.
   * Prefer formulations like:

     * “a body color similar to that of the bird in Image 1”
     * “wings similar in shape and length to those of the object in Image 1”
     * “a bill similar in color to the bill of the bird in Image 1”
   * Avoid formulations that explicitly repeat the shared attribute values, such as:

     * “both objects have long, narrow wings”
     * “both objects have dark brown bodies”
   * The goal is that the reader must look at the reference bird to know what the shared attribute actually looks like.

4. Differences between reference and target (absolute descriptions)

   * Identify all visual attributes where the target bird differs from the reference bird.
   * Describe these differences using absolute descriptions of the target’s attributes, for example:

     * “its body is white”
     * “it has a dark gray body”
     * “it has a hooked bill”
     * “it has a forked tail”
     * “it has a red throat pouch”
   * These differences should be based ONLY on what is explicitly written in the target caption (and, when relevant, by contrast with what is written in the reference caption).
   * Do NOT add new attributes that are not mentioned in the text.
   * List no more than three of the most major of these differences in the final caption.

5. Check against distractor captions

   * Compare your current similarity-and-difference description of the target against the captions of the other distractor objects.
   * If your description could also match one of the distractor objects (because it shares many of the same attributes), then your caption may not be discriminative enough.
   * In that case, look for additional attributes in the target caption that:

     * Are explicitly mentioned for the target, AND
     * Help distinguish it from the overlapping distractor(s).
   * Add one or more of these distinguishing attributes to your caption, still following the rules above:

     * Similarities must remain relational to the object in Image 1.
     * Differences must be absolute descriptions of the target’s explicitly mentioned attributes.

6. Limited and careful use of negation

   * You may use negation (“does not have…”) ONLY if:

     * The negated feature is explicitly mentioned in a distractor caption, AND
     * Negation is necessary to distinguish the target from that object.
   * Do NOT introduce negation about attributes that are not mentioned in any caption.
   * Do NOT rely on negation as the main distinguishing factor if a more direct positive attribute difference is available.

7. Output formatting

   * Write 1–3 sentences of fluent natural language.
   * Refer to the reference object as “the object in Image 1”.
   * Refer to the target object as “the object I am looking for”.
   * Do NOT mention option letters (A, B, C, D).
   * Do NOT mention any class or species names.
   * Do NOT mention the words “reference object”, “target object”, “distractor”, or any internal reasoning.

You can output two sections:
[optional] Reasoning: A step-by-step analysis where you explain your reasoning.
Final caption: The final, clean 1–3 sentence final caption, with no bullet points, no explanations, and no visible reasoning steps.


8. Summary of strict constraints

   * Do NOT invent any new features not explicitly present in the captions.
   * For similarities: describe them ONLY as relations to the object in Image 1 (e.g., “similar to the bird in Image 1”) and avoid repeating the exact attribute values.
   * For differences: describe the target’s attributes with explicit values, but only those present in the text.
   * Ensure the caption is sufficiently specific to distinguish the target from the distractor birds, while still requiring the reference image to interpret the relational similarities.



Here is an example:

Example 1:

Original:
"
"mcq_id": "Black_footed_Albatross",
"difficulty": "Hard",
"options": {
A: "This image shows a Laysan Albatross bird which has a white body, black wings with white underparts, a pale pink bill with a dark tip, and is often seen soaring gracefully over oceans or resting on grassy areas.",
B: "This image shows a Sooty Albatross bird which has a dark gray body, a slender hooked bill, long narrow wings, and a subtle white neck band, distinguishing it from similar seabirds.",
C: "This image shows a Black-footed Albatross bird which has a predominantly dark brown body, a pale head and neck, a long pinkish bill, and long, narrow wings suited for soaring over oceans.",
D: "This image shows a Frigatebird bird which has a sleek black body, long wings, a forked tail, and a distinctive red throat pouch inflated during mating displays, distinguishing it from other seabirds."
},
"Reference_Class": "Black-footed Albatross"
"Target_Class": "Sooty Albatross"

Output:
{
**Reasoning**
**Final caption**
"The object I am looking for has a body of similar color as the body of the bird in Image 1, and similar wing shape and length as the bird in image 1. However, it has a hooked bill and a subtle white neck band."
}

Now apply this to this MCQ Entry:

{input_json}

"""

In [10]:
# [optional] "reasoning": A step-by-step analysis where you explain your reasoning.
# "modification_caption": The final, clean 1–3 sentence final caption, with no bullet points, no explanations, and no visible reasoning steps. 


In [11]:
import time

def parse_llm_response(response: str) -> Dict[str, Any]:
    """Parse the LLM response to extract JSON."""
    # Try to extract JSON from response
    # Some LLMs might wrap it in markdown code blocks
    response = response.strip()
    
    # Remove markdown code blocks if present
    if response.startswith("```json"):
        response = response[7:]
    elif response.startswith("```"):
        response = response[3:]
    
    if response.endswith("```"):
        response = response[:-3]
    
    response = response.strip()
    
    # Parse JSON
    return json.loads(response)


def parse_modification_caption(text: str) -> str:
    """Parse various formats of modification captions and extract the final string.
    
    Handles formats like:
    - **Final caption** followed by text
    - "Final caption": "text"
    - Plain text
    
    Args:
        text: Raw text from LLM response
        
    Returns:
        Cleaned caption string
    """
    # Remove extra whitespace
    text = text.strip()
    
    # Pattern 1: **Final caption** followed by text (with or without quotes)
    pattern1 = r'\*\*Final caption\*\*\s*["\']?(.*?)(?:["\']?\s*\*\*Final caption\*\*|$)'
    match1 = re.search(pattern1, text, re.DOTALL | re.IGNORECASE)
    if match1:
        caption = match1.group(1).strip()
        # Remove trailing quotes if present
        caption = caption.strip('"\'')
        return caption
    
    # Pattern 2: "Final caption": "text"
    pattern2 = r'"Final caption"\s*:\s*"(.*?)"'
    match2 = re.search(pattern2, text, re.IGNORECASE)
    if match2:
        return match2.group(1).strip()
    
    # Pattern 3: JSON format with modification_caption key
    pattern3 = r'"modification_caption"\s*:\s*"(.*?)"'
    match3 = re.search(pattern3, text)
    if match3:
        return match3.group(1).strip()
    
    # Pattern 4: Look for "The bird I am looking for..." sentence
    pattern4 = r'(The object I am looking for.*?)(?:\n|$)'
    match4 = re.search(pattern4, text, re.IGNORECASE)
    if match4:
        caption = match4.group(1).strip()
        # Remove trailing quotes or asterisks
        caption = caption.strip('"\'*')
        return caption
    
    # If no pattern matches, print and return the cleaned text
    print (text)
    return asd


def process_all_questions(input_filepath: str, output_filepath: str):
    """
    Main function to process all questions.
    
    Args:
        input_filepath: Path to input JSON file
        output_filepath: Path to output JSON file
    """
    # Load original questions
    print(f"Loading questions from {input_filepath}...")
    # questions = load_questions(input_filepath)

    # with open(input_filepath, 'r', encoding='utf-8') as f: # claude
    #     questions = json.load(f)

    with open(input_filepath, 'r') as f:
        questions = json.load(f)


    print(f"Loaded {len(questions)} questions")

    captions_mcqid_pair_dict = get_map_dict(questions)
    
    # Process each question
    composed_questions = []
    failed_questions = []
    failed_indices = []
    all_composed_questions = []
    
    for i, question in enumerate(questions):

        # if question['difficulty'] == "Medium":
        #     continue
            
        # print(f"\nProcessing question {i+1}/{len(questions)}: {question['mcq_id']} ({question['difficulty']})")
        
        try:

            
            for option_key in question['options'].keys():

                processed_mcq = remove_similarity_scores(question)
                correct_answer = processed_mcq['correct_answer']

                if (correct_answer == option_key):
                    continue
                
                
                # Step 2: Get class names
                target_caption = processed_mcq['options'][option_key]
                reference_caption = processed_mcq['options'][correct_answer]

                target_class = captions_mcqid_pair_dict[target_caption]
                reference_class = captions_mcqid_pair_dict[reference_caption]

                target_class = target_class.replace("_", " ")
                reference_class = reference_class.replace("_", " ")
                
                # Store class information
                processed_mcq['target_class'] = target_class
                processed_mcq['reference_class'] = reference_class

                # print("============================")
                # print(f"option_key: {option_key}")
                # print(f"target_caption: {target_caption}")
                # print(f"target_class: {target_class}")
                                
                
                # Step 3: Create prompt
                prompt = create_prompt(processed_mcq)

                # Call LLM
                # print("  Calling LLM...")
                llm_response = call_llm(prompt)
                # print(f"\n-----llm_response: {llm_response}\n-----")
                
                # Parse response
                # print("  Parsing response...")
                # parsed_llm_response = parse_llm_response(llm_response)
                # modification_caption = parsed_llm_response['modification_caption']

                modification_caption = parse_modification_caption(llm_response)

                # print(f"\n------------modification_caption: {modification_caption}")
                    
                processed_mcq['modification_caption'] = modification_caption
                processed_mcq['correct_answer'] = option_key
    
                all_composed_questions.append(processed_mcq)
                
                # Validate that key fields are preserved
                if "mcq_id" in processed_mcq:
                    assert processed_mcq['mcq_id'] == question['mcq_id']
                else:
                    processed_mcq['mcq_id'] = question['mcq_id']
                if "difficulty" in processed_mcq:
                    assert processed_mcq['difficulty'] == question['difficulty']
                else:
                    processed_mcq['difficulty'] = question['difficulty']
    
                
                    
    
                
                composed_questions.append(processed_mcq)
                # print (f"\n -------------- final mcq: {processed_mcq}")
            print(f"✓ Success {i}")
            
        except Exception as e:
            print(f"  ✗ Error processing question: {e}")
            print(f"  Skipping this question...")
            print(f"\n-----llm_response: {llm_response}\n-----")
            print(f"\n------------modification_caption: {modification_caption}")
            failed_questions.append(question)
            failed_indices.append(i)
            continue


        time.sleep(1)
    
    # Save results --- Claude
    print(f"\nSaving {len(composed_questions)} composed questions to {output_filepath}...")
    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(composed_questions, f, indent=2, ensure_ascii=False)

    # with open(output_filename, 'w') as f:
    #     json.dump(new_data, f, indent=4)
    # print(f"\n✅ Successfully saved the modified data to {output_filename}")
    
    print("Done!")
    print(f"Successfully processed: {len(composed_questions)}/{len(questions)}")
    return failed_questions, failed_indices, all_composed_questions


# Run Code

In [12]:
# 1. Define the filename
# filename = "F:/abbott/vlm/fine grained robustness vlm benchmark/mcq_json files/new_cub_class_descriptions.json"





# dataset_list = ["new_cub_with_class_descriptions", "new_food_with_class_descriptions", "new_aircraft_with_class_descriptions", "new_car_with_class_descriptions", "new_dogs_with_class_descriptions", "new_descriptors_inaturalist_wo_class_name_hard"]
# new_dataset_list = ["bird", "food", "aircraft", "car", "dogs", "inaturalist"]

# dataset_list = ["new_cub_with_class_descriptions", "new_food_with_class_descriptions", "new_aircraft_with_class_descriptions", "new_car_with_class_descriptions", "new_dogs_with_class_descriptions"]
# new_dataset_list = ["bird", "food", "aircraft", "car", "dogs"]

dataset_list = ["new_car_with_class_descriptions", "new_dogs_with_class_descriptions"]
new_dataset_list = ["car", "dogs"]

for i, dataset_name in enumerate(dataset_list):
    
    INPUT_FILE = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{dataset_name}.json"
    OUTPUT_FILE = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/task_4_composed_questions_{new_dataset_list[i]}.json"

    failed_questions, failed_indices, all_composed_questions = process_all_questions(INPUT_FILE, OUTPUT_FILE)

    print(f"len(failed_questions): {len(failed_questions)}")
    print(f"len(failed_indices): {len(failed_questions)}")


    

Loading questions from /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_with_class_descriptions.json...
Loaded 392 questions
✓ Success 0
✓ Success 1
✓ Success 2
✓ Success 3
✓ Success 4
✓ Success 5
✓ Success 6
✓ Success 7
✓ Success 8
✓ Success 9
✓ Success 10
✓ Success 11
✓ Success 12
✓ Success 13
✓ Success 14
✓ Success 15
✓ Success 16
✓ Success 17
✓ Success 18
✓ Success 19
✓ Success 20
✓ Success 21
✓ Success 22
✓ Success 23
✓ Success 24
✓ Success 25
✓ Success 26
✓ Success 27
✓ Success 28
✓ Success 29
✓ Success 30
✓ Success 31
✓ Success 32
✓ Success 33
✓ Success 34
✓ Success 35
✓ Success 36
✓ Success 37
✓ Success 38
✓ Success 39
✓ Success 40
✓ Success 41
✓ Success 42
✓ Success 43
✓ Success 44
✓ Success 45
✓ Success 46
✓ Success 47
✓ Success 48
✓ Success 49
✓ Success 50
✓ Success 51
✓ Success 52
✓ Success 53
✓ Success 54
✓ Success 55
✓ Success 56
✓ Success 57
✓ Success 58
✓ Success 59
✓ Success 60
✓ Success 61
✓ Success 62
✓ Success 63
✓ Success 64
✓ Success 65
✓ Suc